# Solutions — Comprehensions: list, dict, set, conditional, nested

Exercise notebook: `03_pythonic_python/01_comprehensions.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Active users' emails

**Working solution**

In [ ]:
def active_emails(users):
    return [u["email"].strip().lower() for u in users if u.get("active") and (u.get("email") or "").strip()]

**Explanation**

Filter in the `if` clause, transform in the expression. One line replaces `new ArrayList` + loop + if + add.

**Java-style loop**

In [ ]:
def active_emails_loop(users):
    result = []
    for u in users:
        if u.get("active") == True:
            email = u.get("email")
            if email is not None and email.strip() != "":
                result.append(email.strip().lower())
    return result

**Complexity:** O(n).

**Common mistakes**

- `== True` comparisons — just use truthiness.

## Exercise 2 — 🟡 Medium · Price map with the walrus operator

**Working solution**

In [ ]:
def to_float(text):
    try:
        return float(text)
    except (TypeError, ValueError):
        return None

def price_map(rows):
    return {r["product_id"]: price for r in rows
            if r.get("active", "").lower() == "true" and (price := to_float(r.get("price"))) is not None and price > 0}

**Explanation**

Comprehensions can't contain `try/except`, so push the risky bit into a helper returning `None`. The walrus `:=` binds the
converted value inside the `if` so the output expression can reuse it.

**Complexity:** O(n).

**Common mistakes**

- Calling `to_float` twice (once in the filter, once in the value).
- Forgetting that `float('nan')` parses — fine here since `nan > 0` is False.

## Exercise 3 — 🟡 Medium · Distinct categories per customer

**Working solution**

In [ ]:
from collections import defaultdict

def categories_by_customer(orders):
    by_customer = defaultdict(list)
    for o in orders:
        by_customer[o["customer"]["id"]].append(o)
    result = {cid: sorted({i["category"] for o in os for i in o["items"] if i.get("category")})
              for cid, os in sorted(by_customer.items())}
    return {cid: cats for cid, cats in result.items() if cats}

**Explanation**

Nested comprehension clauses read in the same order as nested loops: `for o in os for i in o["items"]`. A set comprehension
`{...}` dedupes for free.

**Complexity:** O(items + c log c).

**Common mistakes**

- Putting the `for` clauses in the wrong order (inner first).
- Building lists and calling `list(set(...))` — loses sorting.

## Exercise 4 — 🟠 Hard · Schema-driven record normalization

**Working solution**

In [ ]:
def _convert(fn, value):
    if value is None or value == "":
        return None
    try:
        return fn(value)
    except (TypeError, ValueError):
        return None

def normalize(records, schema, rename=None):
    rename = rename or {}
    renamed = ({rename.get(k, k): v for k, v in r.items()} for r in records)
    return [{col: _convert(fn, r.get(col)) for col, fn in schema.items()} for r in renamed]

**Explanation**

Two comprehension stages: rename keys (a generator expression — lazy, no intermediate list), then project+convert.
Using types like `int`/`float`/`str` as the schema's callables is first-class functions in action.

**Complexity:** O(n × columns).

**Common mistakes**

- Mutable default argument `rename={}` — use `None` and replace inside.
- Dropping records with one bad field — usually you want the row with a null + a validation report.

## Challenge — 🔴 Very Hard · Pivot table in pure Python

**Working solution**

In [ ]:
from collections import defaultdict

def pivot(rows, index, columns, values, agg=sum, fill=0):
    cells = defaultdict(list)
    for r in rows:
        cells[(r[index], r[columns])].append(r[values])
    idx_values = sorted({i for i, _ in cells})
    col_values = sorted({c for _, c in cells})
    return {i: {c: agg(cells[(i, c)]) if (i, c) in cells else fill for c in col_values} for i in idx_values}

**Explanation**

Collect values per `(index, column)` tuple, compute the distinct axes with set comprehensions, then build the full grid with a
nested dict comprehension. Passing `agg` as a function (`sum`, `max`, `len`, `statistics.mean`) is the same idea as
`groupBy().agg(...)`. Note: check membership with `in` — reading `cells[(i, c)]` on a defaultdict would *insert* empty lists.

**Complexity:** O(n + I×C).

**Common mistakes**

- Indexing a `defaultdict` to test existence — it silently creates entries.
- Ragged output where rows lack some column keys.